In [1]:
# ===================== NB2 CELL 1 - Setup + run settings =====================
# Settings: Accelerator = GPU T4 x2, Internet = On, Secret "HF" ticked.
# >>> The ONLY line you change between sessions is MODE <<<
MODE = "explore"        # "smoke" (interactive test) -> "explore" (commit) -> "main" (commit)

EXPLORE_TOKENS = 100e6  # per architecture in the exploration runs
MAIN_ARCH      = "llama"
MAIN_RUN       = "main_llama"
MAIN_TOKENS    = 1.05e9
MIX            = "supreme_court:2,acts:0.6,tribunals:0.65,high_courts:0.4"   # epochs over each source

import os, sys, json, time, subprocess, threading, queue, shutil
os.environ["HF_HOME"] = "/tmp/hf_home"
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER   = api.whoami()["name"]
DATA_REPO = f"{HF_USER}/cs781-p1-data"
CKPT_REPO = f"{HF_USER}/cs781-p1-ckpt"
DATA_DIR  = "/tmp/cs781/prepared"
CODE_DIR  = "/kaggle/working"
OUT_DIR   = "/kaggle/working/runs"
os.makedirs(OUT_DIR, exist_ok=True)

import torch
print(f"MODE={MODE} | torch {torch.__version__} | CUDA {torch.version.cuda} | GPUs {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU{i}: {p.name} {p.total_memory/1e9:.1f} GB, compute capability {p.major}.{p.minor}")
print("HF:", HF_USER, "| data:", DATA_REPO, "| checkpoints:", CKPT_REPO)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


MODE=explore | torch 2.11.0+cu128 | CUDA 12.8 | GPUs 2
  GPU0: Tesla T4 15.6 GB, compute capability 7.5
  GPU1: Tesla T4 15.6 GB, compute capability 7.5
HF: Abhayk24 | data: Abhayk24/cs781-p1-data | checkpoints: Abhayk24/cs781-p1-ckpt


In [2]:
# ===================== NB2 CELL 2 - Download the prepared data from our private HF repo =====================
from huggingface_hub import snapshot_download
t0 = time.time()
snapshot_download(DATA_REPO, repo_type="dataset", local_dir=DATA_DIR, token=HF_TOKEN)
meta = json.load(open(f"{DATA_DIR}/meta.json"))
for name, n in meta["token_counts"].items():
    size = os.path.getsize(f"{DATA_DIR}/bin/{name}.bin")
    assert size == 2 * n, f"{name}: size {size} != {2*n}"
    print(f"  {name:22s} {n/1e6:9.2f}M tokens  OK")
print(f"Data ready in {time.time()-t0:.0f}s | vocab {meta['vocab_size']} | dev examples {meta['n_dev']}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

  supreme_court_val           2.49M tokens  OK
  supreme_court_train       250.74M tokens  OK
  acts_val                    1.74M tokens  OK
  acts_train                148.80M tokens  OK
  tribunals_val               4.32M tokens  OK
  tribunals_train           466.99M tokens  OK
  high_courts_val             4.25M tokens  OK
  high_courts_train         454.55M tokens  OK
Data ready in 6s | vocab 32768 | dev examples 479


cell 3


In [3]:
%%writefile /kaggle/working/model.py
"""CS-781 Phase 1 - four legal language-model architectures, written from scratch in PyTorch.

  gpt2    : classic decoder-only Transformer (learned absolute positions, LayerNorm, GELU MLP, multi-head attention)
  llama   : modern decoder-only Transformer (RoPE, RMSNorm, SwiGLU, grouped-query attention, no biases)
  shared  : lightweight decoder (ALBERT-style factorised embeddings + cycle cross-layer weight sharing + multi-query attn)
  encdec  : encoder-decoder Transformer (bidirectional encoder over the prompt, causal decoder with cross-attention)

All models: tied input/output embeddings, pre-norm residual blocks, PyTorch SDPA attention, KV-cache generation.
"""
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    arch: str = "llama"
    vocab_size: int = 32768
    d_model: int = 768
    n_layer: int = 12            # decoder-only: layers; encdec: layers in EACH of encoder and decoder; shared: effective depth
    n_head: int = 12
    n_kv_head: int = 4           # GQA groups (llama/encdec); shared uses 1 (MQA); gpt2 uses n_head (MHA)
    d_ff: int = 2304
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    dropout: float = 0.0
    n_unique_layers: int = 6     # shared only: number of distinct blocks, reused cyclically
    d_embed: int = 256           # shared only: factorised embedding size
    norm_eps: float = 1e-5

    def to_dict(self):
        return asdict(self)


def preset(arch: str, **overrides) -> ModelConfig:
    base = {
        "gpt2":   dict(arch="gpt2", d_model=768, n_layer=12, n_head=12, n_kv_head=12, d_ff=3072),
        "llama":  dict(arch="llama", d_model=768, n_layer=12, n_head=12, n_kv_head=4, d_ff=2304),
        "shared": dict(arch="shared", d_model=768, n_layer=12, n_head=12, n_kv_head=1, d_ff=3072,
                       n_unique_layers=6, d_embed=256),
        "encdec": dict(arch="encdec", d_model=768, n_layer=6, n_head=12, n_kv_head=4, d_ff=2304),
    }[arch]
    base.update(overrides)
    return ModelConfig(**base)


# ----------------------------------------------------------------------------------------------------------------
# building blocks
# ----------------------------------------------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.ln = nn.LayerNorm(d, eps=eps)

    def forward(self, x):
        return F.layer_norm(x.float(), self.ln.normalized_shape, self.ln.weight.float(), self.ln.bias.float(),
                            self.ln.eps).to(x.dtype)


def rope_cache(head_dim, max_len, theta, device=None):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv)                       # (T, hd/2)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (T, hd/2) - rotate pairs (x[..., :hd/2], x[..., hd/2:])
    xf = x.float()
    x1, x2 = xf[..., : xf.shape[-1] // 2], xf[..., xf.shape[-1] // 2:]
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.to(x.dtype)


class Attention(nn.Module):
    """Multi-head / grouped-query / multi-query attention. Self-attention (optionally causal) or cross-attention."""

    def __init__(self, cfg: ModelConfig, n_kv_head: int, bias: bool, use_rope: bool, cross: bool = False):
        super().__init__()
        d, h = cfg.d_model, cfg.n_head
        assert d % h == 0 and h % n_kv_head == 0
        self.h, self.kv, self.hd = h, n_kv_head, d // h
        self.use_rope, self.cross = use_rope, cross
        self.q = nn.Linear(d, h * self.hd, bias=bias)
        self.k = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.v = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.o = nn.Linear(h * self.hd, d, bias=bias)
        self.o.is_residual_proj = True
        self.dropout = cfg.dropout

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.hd).transpose(1, 2)          # (B, n, T, hd)

    def _expand(self, x):
        return x if self.kv == self.h else x.repeat_interleave(self.h // self.kv, dim=1)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None):
        """
        x: (B, T, d).  rope: (cos, sin) full tables.  pos: absolute position of x[:,0] (for RoPE during decoding).
        cache: dict used during generation. Self-attn: keys 'k','v' accumulate. Cross-attn: 'k','v' computed once.
        memory: encoder output (B, S, d) for cross-attention.
        """
        q = self._split(self.q(x), self.h)
        if self.cross:
            if cache is not None and "k" in cache:
                k, v = cache["k"], cache["v"]
            else:
                k, v = self._split(self.k(memory), self.kv), self._split(self.v(memory), self.kv)
                if cache is not None:
                    cache["k"], cache["v"] = k, v
            is_causal = False
        else:
            k, v = self._split(self.k(x), self.kv), self._split(self.v(x), self.kv)
            if self.use_rope:
                cos, sin = rope
                T = x.shape[1]
                q = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T])
                k = apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
            if cache is not None:
                if "k" in cache:
                    k = torch.cat([cache["k"], k], dim=2); v = torch.cat([cache["v"], v], dim=2)
                cache["k"], cache["v"] = k, v
            # causal mask only needed when queries and keys cover the same span (training / prefill)
            is_causal = causal and q.shape[2] == k.shape[2] and q.shape[2] > 1
            if causal and q.shape[2] != k.shape[2] and q.shape[2] != 1:
                raise ValueError("multi-token decode with a non-empty cache is not supported")
        y = F.scaled_dot_product_attention(q, self._expand(k), self._expand(v), is_causal=is_causal,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(x.shape[0], x.shape[1], self.h * self.hd)
        return self.o(y)


class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig, kind: str, bias: bool):
        super().__init__()
        self.kind = kind
        if kind == "swiglu":
            self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        else:
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=bias)
        self.down.is_residual_proj = True
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        if self.kind == "swiglu":
            h = F.silu(self.gate(x)) * self.up(x)
        else:
            h = F.gelu(self.up(x), approximate="tanh")
        return self.drop(self.down(h))


class Block(nn.Module):
    """Pre-norm Transformer block: x + Attn(Norm(x)) [+ CrossAttn(Norm(x), memory)] + MLP(Norm(x))."""

    def __init__(self, cfg: ModelConfig, style: str, n_kv_head: int, cross: bool = False):
        super().__init__()
        gpt2 = style == "gpt2"
        Norm = LayerNorm if gpt2 else RMSNorm
        self.n1 = Norm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg, n_kv_head, bias=gpt2, use_rope=not gpt2)
        self.has_cross = cross
        if cross:
            self.nc = Norm(cfg.d_model, cfg.norm_eps)
            self.xattn = Attention(cfg, n_kv_head, bias=False, use_rope=False, cross=True)
        self.n2 = Norm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg, "gelu" if gpt2 else "swiglu", bias=gpt2)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None, xcache=None):
        x = x + self.drop(self.attn(self.n1(x), rope=rope, pos=pos, causal=causal, cache=cache))
        if self.has_cross:
            x = x + self.drop(self.xattn(self.nc(x), memory=memory, cache=xcache))
        return x + self.mlp(self.n2(x))


# ----------------------------------------------------------------------------------------------------------------
# the language model
# ----------------------------------------------------------------------------------------------------------------
class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        a = cfg.arch
        assert a in ("gpt2", "llama", "shared", "encdec"), a
        d = cfg.d_model
        if a == "shared":
            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_embed)
            self.emb_proj = nn.Linear(cfg.d_embed, d, bias=False)          # E -> d   (factorised embedding)
        else:
            self.tok_emb = nn.Embedding(cfg.vocab_size, d)
        if a == "gpt2":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, d)
        self.drop = nn.Dropout(cfg.dropout)

        if a == "gpt2":
            self.blocks = nn.ModuleList([Block(cfg, "gpt2", cfg.n_head) for _ in range(cfg.n_layer)])
        elif a == "llama":
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
        elif a == "shared":
            assert cfg.n_layer % cfg.n_unique_layers == 0
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_unique_layers)])
        else:  # encdec
            self.enc_blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
            self.enc_norm = RMSNorm(d, cfg.norm_eps)
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head, cross=True) for _ in range(cfg.n_layer)])
        self.norm = LayerNorm(d, cfg.norm_eps) if a == "gpt2" else RMSNorm(d, cfg.norm_eps)

        if a != "gpt2":
            hd = d // cfg.n_head
            cos, sin = rope_cache(hd, max(2 * cfg.max_seq_len, 2048), cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        n_res = 2 * self.n_layer_applications() + (cfg.n_layer if a == "encdec" else 0)
        for name, p in self.named_parameters():
            if name.endswith("o.weight") or name.endswith("down.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    # -- helpers ------------------------------------------------------------------------------------------------
    def n_layer_applications(self):
        return self.cfg.n_layer * (2 if self.cfg.arch == "encdec" else 1)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _rope(self):
        return None if self.cfg.arch == "gpt2" else (self.rope_cos, self.rope_sin)

    def _embed(self, idx, pos=0):
        x = self.tok_emb(idx)
        if self.cfg.arch == "shared":
            x = self.emb_proj(x)
        if self.cfg.arch == "gpt2":
            T = idx.shape[1]
            if pos + T > self.cfg.max_seq_len:
                raise ValueError(f"gpt2 positions exceed max_seq_len ({pos + T} > {self.cfg.max_seq_len})")
            x = x + self.pos_emb(torch.arange(pos, pos + T, device=idx.device))[None]
        return self.drop(x)

    def _logits(self, h):
        h = self.norm(h)
        if self.cfg.arch == "shared":
            h = F.linear(h, self.emb_proj.weight.t())     # d -> E   (tied with the input projection)
        return F.linear(h, self.tok_emb.weight)            # tied output embedding

    def _decoder_layers(self):
        if self.cfg.arch == "shared":                       # cycle sharing: 0,1,..,5,0,1,..,5
            return [self.blocks[i % self.cfg.n_unique_layers] for i in range(self.cfg.n_layer)]
        return list(self.blocks)

    def encode(self, src):
        x = self._embed(src)
        for blk in self.enc_blocks:
            x = blk(x, rope=self._rope(), pos=0, causal=False)
        return self.enc_norm(x)

    # -- training forward -----------------------------------------------------------------------------------------
    def forward(self, idx, targets=None, src=None, loss_mask=None):
        """Decoder-only: idx (B,T) -> logits; loss vs targets (B,T).  encdec: src (B,S) is the encoder input."""
        memory = self.encode(src) if self.cfg.arch == "encdec" else None
        x = self._embed(idx)
        for blk in self._decoder_layers():
            x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
        logits = self._logits(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
        if loss_mask is not None:
            m = loss_mask.reshape(-1).float()
            return logits, (loss * m).sum() / m.sum().clamp(min=1)
        return logits, loss.mean()

    # -- generation -----------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, prompt, max_new_tokens, temperature=1.0, top_k=0, top_p=1.0, repetition_penalty=1.0,
                 no_repeat_ngram=0, banned_ids=(), bos_id=2, generator=None, use_cache=True):
        """prompt: (B, T) LongTensor, all rows the same length (no padding). Returns (B, max_new_tokens) new ids.
        temperature 0 = greedy."""
        B = prompt.shape[0]
        dev = prompt.device
        layers = self._decoder_layers()
        if self.cfg.arch == "encdec":
            memory = self.encode(prompt)
            seq = torch.full((B, 1), bos_id, dtype=torch.long, device=dev)   # decoder starts from <|bos|>
            history = prompt                                                   # for repetition penalties
        else:
            memory, seq, history = None, prompt, prompt
        caches = [dict() for _ in layers] if use_cache else None
        xcaches = [dict() for _ in layers] if (use_cache and memory is not None) else [None] * len(layers)
        out = []
        cur, pos = seq, 0
        for step in range(max_new_tokens):
            if use_cache:
                x = self._embed(cur, pos=pos)
                for li, blk in enumerate(layers):
                    x = blk(x, rope=self._rope(), pos=pos, causal=True, cache=caches[li], memory=memory,
                            xcache=xcaches[li])
                pos += cur.shape[1]
            else:
                x = self._embed(seq)
                for blk in layers:
                    x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
            logits = self._logits(x[:, -1:, :])[:, -1, :].float()
            full_hist = torch.cat([history] + out, dim=1) if out else history
            nxt = sample_next(logits, full_hist, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram,
                              banned_ids, generator)
            out.append(nxt)
            seq = torch.cat([seq, nxt], dim=1)
            cur = nxt
        return torch.cat(out, dim=1)


def sample_next(logits, history, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram, banned_ids,
                generator=None):
    """logits (B,V) float32, history (B,L) long (prompt + generated so far). Returns (B,1)."""
    logits = logits.clone()
    if banned_ids:
        logits[:, list(banned_ids)] = -float("inf")
    if repetition_penalty != 1.0:
        # CTRL-style penalty on tokens already present in the recent history (last 256 tokens)
        recent = history[:, -256:]
        sc = torch.gather(logits, 1, recent)
        sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
        logits.scatter_(1, recent, sc)
    if no_repeat_ngram and history.shape[1] >= no_repeat_ngram:
        n = no_repeat_ngram
        hist = history[:, -512:].tolist()
        for b, h in enumerate(hist):
            prefix = tuple(h[-(n - 1):]) if n > 1 else tuple()
            banned = {tuple(h[i:i + n])[-1] for i in range(len(h) - n + 1) if tuple(h[i:i + n - 1]) == prefix}
            if banned:
                logits[b, list(banned)] = -float("inf")
    if temperature <= 0:
        return logits.argmax(dim=-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = torch.softmax(sl, dim=-1).cumsum(dim=-1)
        remove = cp - torch.softmax(sl, dim=-1) > top_p          # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, si, sl)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1, generator=generator)

Writing /kaggle/working/model.py


In [4]:
%%writefile /kaggle/working/train.py
"""CS-781 Phase 1 trainer: from-scratch legal LM pre-training with deterministic data order, exact resume,
time-limited sessions and checkpoint upload to a private Hugging Face repo.

Single GPU :  python train.py --run_name X --arch llama ...
Two GPUs   :  torchrun --standalone --nproc_per_node=2 train.py --run_name X ...
Rerunning the SAME command resumes from the latest checkpoint (local first, then Hugging Face).
"""
import argparse, json, math, os, sys, time
from contextlib import nullcontext
from datetime import timedelta

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import LegalLM, preset, ModelConfig

SOURCES = ["supreme_court", "acts", "tribunals", "high_courts"]


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--run_name", required=True)
    p.add_argument("--arch", default="llama", choices=["gpt2", "llama", "shared", "encdec"])
    p.add_argument("--model_overrides", default="{}", help='JSON, e.g. {"n_layer": 14}')
    p.add_argument("--data_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="", help="private HF model repo for checkpoints ('' = local only)")
    p.add_argument("--seq_len", type=int, default=1024)
    p.add_argument("--micro_bs", type=int, default=8)
    p.add_argument("--batch_tokens", type=int, default=131072, help="tokens per optimizer step (all GPUs)")
    p.add_argument("--max_tokens", type=float, default=1e9, help="training token budget")
    p.add_argument("--mix", default="supreme_court:2,acts:0.5,tribunals:0.5,high_courts:0.35",
                   help="epochs over each source's training chunks")
    p.add_argument("--lr", type=float, default=6e-4)
    p.add_argument("--min_lr_ratio", type=float, default=0.1)
    p.add_argument("--warmup_steps", type=int, default=-1, help="-1 = 4%% of steps (min 50)")
    p.add_argument("--weight_decay", type=float, default=0.1)
    p.add_argument("--beta1", type=float, default=0.9)
    p.add_argument("--beta2", type=float, default=0.95)
    p.add_argument("--grad_clip", type=float, default=1.0)
    p.add_argument("--seed", type=int, default=1337)
    p.add_argument("--enc_split", default="256:896", help="encdec: encoder length range when splitting chunks")
    p.add_argument("--eval_every", type=int, default=200)
    p.add_argument("--val_chunks", type=int, default=40, help="validation chunks per source")
    p.add_argument("--log_every", type=int, default=20)
    p.add_argument("--ckpt_minutes", type=float, default=45)
    p.add_argument("--time_limit_hours", type=float, default=11.0)
    p.add_argument("--compile", type=int, default=0)
    p.add_argument("--max_steps", type=int, default=-1, help="debug: stop after this many steps")
    return p.parse_args(argv)


# ---------------------------------------------------------------------------------------------------------------
# Hugging Face helpers (all failures are logged, never fatal)
# ---------------------------------------------------------------------------------------------------------------
def hf_upload(repo, local, remote, log):
    if not repo:
        return
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get("HF_TOKEN"))
    for attempt in range(4):
        try:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=repo, repo_type="model",
                            commit_message=f"upload {remote}")
            return True
        except Exception as e:
            log(f"[hf] upload of {remote} failed (attempt {attempt + 1}): {e}")
            time.sleep(20 * (attempt + 1))
    return False


def hf_download(repo, remote, local_dir, log):
    if not repo:
        return None
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download(repo, remote, repo_type="model", local_dir=local_dir,
                               token=os.environ.get("HF_TOKEN"))
    except Exception as e:
        log(f"[hf] no {remote} on the hub ({type(e).__name__})")
        return None


# ---------------------------------------------------------------------------------------------------------------
# data
# ---------------------------------------------------------------------------------------------------------------
class ChunkData:
    """Training data = fixed (source, chunk) order, fully determined by (seed, mix). Batch for (step, micro, rank)
    is a pure function of those indices, so a resumed run continues exactly where it stopped."""

    def __init__(self, data_dir, seq_len, mix, seed):
        self.T = seq_len
        self.arrs, self.n_chunks = {}, {}
        for s in SOURCES:
            a = np.memmap(os.path.join(data_dir, "bin", f"{s}_train.bin"), dtype=np.uint16, mode="r")
            self.arrs[s] = a
            self.n_chunks[s] = (len(a) - 1) // seq_len
        rng = np.random.default_rng(seed)
        parts = []
        for si, s in enumerate(SOURCES):
            w, n = mix.get(s, 0.0), self.n_chunks[s]
            full, frac = int(w), w - int(w)
            for _ in range(full):
                parts.append(np.stack([np.full(n, si, dtype=np.int64), rng.permutation(n)], 1))
            if frac > 1e-9:
                k = int(round(n * frac))
                parts.append(np.stack([np.full(k, si, dtype=np.int64), rng.permutation(n)[:k]], 1))
        order = np.concatenate(parts)
        rng.shuffle(order)
        self.order = order
        self.src_counts = {s: int((order[:, 0] == i).sum()) for i, s in enumerate(SOURCES)}

    def rows(self, start, n):
        out = np.empty((n, self.T + 1), dtype=np.int64)
        for j, (si, ci) in enumerate(self.order[start:start + n]):
            a = self.arrs[SOURCES[si]]
            out[j] = a[ci * self.T: ci * self.T + self.T + 1]
        return out


def load_val(data_dir, seq_len, per_source):
    val = {}
    for s in SOURCES:
        path = os.path.join(data_dir, "bin", f"{s}_val.bin")
        if not os.path.exists(path) or os.path.getsize(path) < 2 * (seq_len + 1):
            continue
        a = np.memmap(path, dtype=np.uint16, mode="r")
        n = (len(a) - 1) // seq_len
        idx = np.linspace(0, n - 1, min(per_source, n)).astype(np.int64)
        val[s] = np.stack([np.asarray(a[i * seq_len: i * seq_len + seq_len + 1], dtype=np.int64) for i in idx])
    return val


def load_dev(data_dir, tokenizer_path):
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(tokenizer_path)
    prompts = json.load(open(os.path.join(data_dir, "dev.json")))
    truth = json.load(open(os.path.join(data_dir, "dev_truth.json")))
    dev = []
    for k in prompts:
        p = tok.encode(prompts[k]["text"], add_special_tokens=False).ids
        r = tok.encode(" " + truth[k]["text"], add_special_tokens=False).ids
        dev.append((np.array(p, dtype=np.int64), np.array(r, dtype=np.int64)))
    return dev


# ---------------------------------------------------------------------------------------------------------------
def main(argv=None):
    args = parse_args(argv)
    ddp = int(os.environ.get("WORLD_SIZE", "1")) > 1
    if ddp:
        backend = "nccl" if torch.cuda.is_available() else "gloo"
        dist.init_process_group(backend=backend, timeout=timedelta(minutes=60))
        rank, world, local_rank = dist.get_rank(), dist.get_world_size(), int(os.environ["LOCAL_RANK"])
    else:
        rank, world, local_rank = 0, 1, 0
    master = rank == 0
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)
    else:
        device = torch.device("cpu")
    run_dir = os.path.join(args.out_dir, args.run_name)
    os.makedirs(run_dir, exist_ok=True)
    log_path = os.path.join(run_dir, "log.jsonl")

    def log(msg):
        if master:
            print(f"[{time.strftime('%H:%M:%S')}][{args.run_name}] {msg}", flush=True)

    def log_json(rec):
        if master:
            with open(log_path, "a") as f:
                f.write(json.dumps(rec) + "\n")

    t_start = time.time()
    torch.manual_seed(args.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    # ---- data & steps ----
    mix = {kv.split(":")[0]: float(kv.split(":")[1]) for kv in args.mix.split(",") if kv}
    for s in mix:
        assert s in SOURCES, s
    data = ChunkData(args.data_dir, args.seq_len, mix, args.seed)
    T, B = args.seq_len, args.micro_bs
    assert args.batch_tokens % (B * T * world) == 0, "batch_tokens must be divisible by micro_bs*seq_len*world"
    accum = args.batch_tokens // (B * T * world)
    rows_per_step = B * accum * world
    total_steps = min(len(data.order) // rows_per_step, int(args.max_tokens // args.batch_tokens))
    warmup = args.warmup_steps if args.warmup_steps >= 0 else max(50, int(0.04 * total_steps))
    log(f"world={world} micro_bs={B} accum={accum} tokens/step={args.batch_tokens:,} total_steps={total_steps:,} "
        f"(= {total_steps * args.batch_tokens / 1e9:.3f}B tokens) warmup={warmup}")
    log(f"chunks available per source: {data.n_chunks} | chunks in mixed order: {data.src_counts}")

    # ---- model ----
    overrides = json.loads(args.model_overrides)
    cfg = preset(args.arch, max_seq_len=T, **overrides)
    raw = LegalLM(cfg).to(device)
    log(f"arch={args.arch} params={raw.num_params() / 1e6:.2f}M config={cfg.to_dict()}")
    assert 50e6 <= raw.num_params() <= 300e6 or os.environ.get("ALLOW_ANY_SIZE") == "1", "model outside 50M-300M"

    cuda = device.type == "cuda"
    if cuda:
        amp_dtype = torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16
        autocast = lambda: torch.autocast("cuda", dtype=amp_dtype)
    else:
        amp_dtype = torch.float32
        autocast = nullcontext
    scaler = torch.amp.GradScaler("cuda", enabled=(cuda and amp_dtype == torch.float16))

    decay = [p for n, p in raw.named_parameters() if p.dim() >= 2]
    no_decay = [p for n, p in raw.named_parameters() if p.dim() < 2]
    groups = [{"params": decay, "weight_decay": args.weight_decay}, {"params": no_decay, "weight_decay": 0.0}]
    try:
        opt = torch.optim.AdamW(groups, lr=args.lr, betas=(args.beta1, args.beta2), eps=1e-8, fused=cuda)
    except Exception:
        opt = torch.optim.AdamW(groups, lr=args.lr, betas=(args.beta1, args.beta2), eps=1e-8)

    # ---- resume ----
    step, best_dev, tokens_seen = 0, float("inf"), 0
    last_path = os.path.join(run_dir, "last.pt")
    if master and not os.path.exists(last_path):
        got = hf_download(args.hf_repo, f"runs/{args.run_name}/last.pt", args.out_dir, log)
        if got:
            hf_download(args.hf_repo, f"runs/{args.run_name}/log.jsonl", args.out_dir, log)
            hf_download(args.hf_repo, f"runs/{args.run_name}/best_model.pt", args.out_dir, log)
    if ddp:
        dist.barrier()
    hf_last = os.path.join(args.out_dir, "runs", args.run_name, "last.pt")   # where hf_hub_download puts it
    if not os.path.exists(last_path) and os.path.exists(hf_last):
        for fn in ["last.pt", "log.jsonl", "best_model.pt"]:
            src = os.path.join(args.out_dir, "runs", args.run_name, fn)
            if master and os.path.exists(src):
                os.replace(src, os.path.join(run_dir, fn))
        if ddp:
            dist.barrier()
    if os.path.exists(last_path):
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        assert ck["config"] == cfg.to_dict(), "checkpoint config differs from requested config"
        assert ck["data_sig"] == [args.seed, args.mix, T, args.batch_tokens, world], \
            f"data order signature changed: {ck['data_sig']} - resume with the same mix/seed/batch/world size"
        raw.load_state_dict(ck["model"])
        opt.load_state_dict(ck["optimizer"])
        if ck.get("scaler") and scaler.is_enabled():
            scaler.load_state_dict(ck["scaler"])
        step, best_dev, tokens_seen = ck["step"], ck["best_dev"], ck["tokens_seen"]
        log(f"RESUMED from step {step} (best dev loss {best_dev:.4f})")
        del ck

    use_compile = bool(args.compile) and args.arch != "encdec"   # encdec: variable split lengths -> no compile
    if args.compile and not use_compile:
        log("torch.compile disabled for encdec (variable sequence lengths would trigger recompiles)")
    model = torch.compile(raw) if use_compile else raw
    if ddp:
        model = DDP(model, device_ids=[local_rank] if cuda else None)

    # ---- evaluation data (rank 0 only) ----
    val = load_val(args.data_dir, T, args.val_chunks) if master else {}
    dev = load_dev(args.data_dir, os.path.join(args.data_dir, "tokenizer.json")) if master else []
    enc_lo, enc_hi = [int(x) for x in args.enc_split.split(":")]
    BOS = 2

    def lr_at(s):
        if s < warmup:
            return args.lr * (s + 1) / warmup
        prog = min(1.0, (s - warmup) / max(1, total_steps - warmup))
        return args.lr * (args.min_lr_ratio + (1 - args.min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * prog)))

    def make_inputs(chunk, split_at):
        """chunk (n, T+1) int64 numpy -> model kwargs on device."""
        x = torch.from_numpy(chunk)
        if cuda:
            x = x.pin_memory().to(device, non_blocking=True)
        else:
            x = x.to(device)
        if args.arch == "encdec":
            src, tgt = x[:, :split_at], x[:, split_at:]
            dec_in = torch.cat([torch.full_like(tgt[:, :1], BOS), tgt[:, :-1]], dim=1)
            return dict(idx=dec_in, targets=tgt, src=src)
        return dict(idx=x[:, :-1], targets=x[:, 1:])

    @torch.no_grad()
    def evaluate():
        raw.eval()
        res = {}
        with autocast():
            for s, chunks in val.items():
                losses = []
                for i in range(0, len(chunks), B):
                    kw = make_inputs(chunks[i:i + B], (3 * T) // 4)   # encdec: fixed 75/25 split
                    _, l = raw(**kw)
                    losses.append(l.item() * len(chunks[i:i + B]))
                res[f"val_{s}"] = sum(losses) / len(chunks)
            tot, n = 0.0, 0
            for p, r in dev:                      # task-aligned: loss of the 128-word reference given the prompt
                r = r[: T // 2]                   # (no-op at T=1024: a 128-word reference is ~155 tokens)
                if args.arch == "encdec":
                    src = torch.from_numpy(p[-enc_hi:])[None].to(device)
                    tgt = torch.from_numpy(r)[None].to(device)
                    dec_in = torch.cat([torch.full_like(tgt[:, :1], BOS), tgt[:, :-1]], dim=1)
                    _, l = raw(dec_in, tgt, src=src)
                else:
                    keep = min(len(p), T - len(r) + 1)
                    seq = torch.from_numpy(np.concatenate([p[-keep:], r]))[None].to(device)
                    mask = torch.zeros_like(seq[:, 1:])
                    mask[:, -len(r):] = 1
                    _, l = raw(seq[:, :-1], seq[:, 1:], loss_mask=mask)
                tot += l.item() * len(r); n += len(r)
        if val:
            res["val_avg"] = float(np.mean([res[f"val_{s}"] for s in val]))
        if n:
            res["dev_loss"] = tot / n
            res["dev_ppl"] = math.exp(res["dev_loss"])
        raw.train()
        return res

    def save_full(path):
        tmp = path + ".tmp"
        torch.save({"model": raw.state_dict(), "optimizer": opt.state_dict(),
                    "scaler": scaler.state_dict() if scaler.is_enabled() else None,
                    "step": step, "best_dev": best_dev, "tokens_seen": tokens_seen, "config": cfg.to_dict(),
                    "args": vars(args), "data_sig": [args.seed, args.mix, T, args.batch_tokens, world]}, tmp)
        os.replace(tmp, path)

    def save_weights(path, metrics):
        tmp = path + ".tmp"
        torch.save({"model": raw.state_dict(), "config": cfg.to_dict(), "step": step, "tokens_seen": tokens_seen,
                    "metrics": metrics, "args": vars(args)}, tmp)
        os.replace(tmp, path)

    def checkpoint(upload_best):
        save_full(last_path)
        hf_upload(args.hf_repo, last_path, f"runs/{args.run_name}/last.pt", log)
        if os.path.exists(log_path):
            hf_upload(args.hf_repo, log_path, f"runs/{args.run_name}/log.jsonl", log)
        if upload_best and os.path.exists(os.path.join(run_dir, "best_model.pt")):
            hf_upload(args.hf_repo, os.path.join(run_dir, "best_model.pt"), f"runs/{args.run_name}/best_model.pt", log)
        log(f"checkpoint saved at step {step}")

    if master and step == 0:
        with open(os.path.join(run_dir, "config.json"), "w") as f:
            json.dump({"model": cfg.to_dict(), "args": vars(args), "params": raw.num_params(),
                       "total_steps": total_steps, "accum": accum}, f, indent=2)
        hf_upload(args.hf_repo, os.path.join(run_dir, "config.json"), f"runs/{args.run_name}/config.json", log)

    if step >= total_steps:
        log("run already complete - nothing to do")
    model.train()
    last_ckpt_t, best_dirty, last_eval = time.time(), False, (None, None)
    t_log, tok_log = time.time(), 0
    stop = torch.zeros(1, device=device)
    while step < total_steps:
        lr = lr_at(step)
        for g in opt.param_groups:
            g["lr"] = lr
        loss_acc = 0.0
        for m in range(accum):
            start = ((step * accum + m) * world + rank) * B
            split_at = 0
            if args.arch == "encdec":
                split_at = int(np.random.default_rng(args.seed * 1_000_003 + step * accum + m).integers(enc_lo, enc_hi + 1))
            kw = make_inputs(data.rows(start, B), split_at)
            sync = (m == accum - 1) or not ddp
            ctx = model.no_sync() if (ddp and not sync) else nullcontext()
            with ctx:
                with autocast():
                    _, loss = model(**kw)
                scaler.scale(loss / accum).backward()
            loss_acc += loss.item() / accum
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw.parameters(), args.grad_clip).item()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        step += 1
        tokens_seen += args.batch_tokens
        tok_log += args.batch_tokens

        if master and (step % args.log_every == 0 or step == 1):
            dt = time.time() - t_log
            tps = tok_log / max(dt, 1e-9)
            eta = (total_steps - step) * args.batch_tokens / max(tps, 1e-9) / 3600
            mem = torch.cuda.max_memory_allocated(device) / 1e9 if cuda else 0
            log(f"step {step}/{total_steps} loss {loss_acc:.4f} lr {lr:.2e} gnorm {gnorm:.2f} "
                f"{tps:,.0f} tok/s mem {mem:.1f}GB elapsed {(time.time() - t_start) / 3600:.2f}h ETA {eta:.2f}h")
            log_json({"step": step, "tokens": tokens_seen, "loss": loss_acc, "lr": lr, "gnorm": gnorm, "tok_s": tps,
                      "time_h": (time.time() - t_start) / 3600})
            t_log, tok_log = time.time(), 0

        if step % args.eval_every == 0 or step == total_steps:
            if master:
                te = time.time()
                res = evaluate()
                res.update(step=step, tokens=tokens_seen, eval_seconds=round(time.time() - te, 1))
                log("EVAL " + " ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in res.items()))
                log_json({"eval": res})
                last_eval = (step, res)
                if res.get("dev_loss", float("inf")) < best_dev:
                    best_dev = res["dev_loss"]
                    save_weights(os.path.join(run_dir, "best_model.pt"), res)
                    best_dirty = True
                    log(f"new best dev loss {best_dev:.4f}")
            if ddp:
                dist.barrier()

        # synchronised stop / checkpoint decision (rank 0 decides, everyone follows)
        if master:
            over = (time.time() - t_start) / 3600 > args.time_limit_hours
            stop[0] = 1.0 if (over or (args.max_steps > 0 and step >= args.max_steps)) else 0.0
        if ddp:
            dist.broadcast(stop, src=0)
        if master and (time.time() - last_ckpt_t > args.ckpt_minutes * 60 or stop.item() > 0):
            checkpoint(best_dirty)
            best_dirty, last_ckpt_t = False, time.time()
        if stop.item() > 0:
            log(f"STOPPING at step {step}/{total_steps} (time/step limit). Rerun the same command to resume.")
            break

    if master and step >= total_steps:
        res = last_eval[1] if last_eval[0] == step else evaluate()
        save_weights(os.path.join(run_dir, "final_model.pt"), res)
        checkpoint(best_dirty)
        hf_upload(args.hf_repo, os.path.join(run_dir, "final_model.pt"), f"runs/{args.run_name}/final_model.pt", log)
        if os.path.exists(os.path.join(run_dir, "best_model.pt")):
            hf_upload(args.hf_repo, os.path.join(run_dir, "best_model.pt"), f"runs/{args.run_name}/best_model.pt", log)
        with open(os.path.join(run_dir, "DONE"), "w") as f:
            f.write(json.dumps(res))
        hf_upload(args.hf_repo, os.path.join(run_dir, "DONE"), f"runs/{args.run_name}/DONE", log)
        log(f"TRAINING COMPLETE. final metrics: {res}")
    if ddp:
        dist.barrier()
        dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/train.py


In [5]:
# ===================== NB2 CELL 5 (v2) - remaining GPU checks (runs only when MODE == "smoke") =====================
# Speed of the 4 architectures is already measured. This checks: evaluation + HF upload, resume FROM HF,
# 2-GPU training, and (optional) torch.compile speed-up. ~10 minutes.
def run(cmd, env_extra=None, required=True,
        keep=("params=", "world=", "step ", "EVAL", "checkpoint", "RESUMED", "STOPPING", "best", "compile",
              "Error", "error", "Traceback", "[hf]")):
    env = dict(os.environ, **(env_extra or {}))
    print("\n>>>", cmd[:170], "...", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, env=env, capture_output=True, text=True)
    out = r.stdout + r.stderr
    for line in out.splitlines():
        if any(k in line for k in keep):
            print("   ", line[:230])
    print(f"    (exit code {r.returncode}, {time.time()-t0:.0f}s)")
    if r.returncode != 0:
        print(out[-3000:])
        if required:
            raise RuntimeError("required smoke check failed - paste this output to Claude")
        print("    -> OPTIONAL check failed: fine, we simply won't use this option.")
    return r.returncode == 0

if MODE == "smoke":
    SM = "/tmp/smoke"; shutil.rmtree(SM, ignore_errors=True)
    py = f"{sys.executable} {CODE_DIR}/train.py --data_dir {DATA_DIR} --out_dir {SM}"
    ddp = f"{sys.executable} -m torch.distributed.run --standalone --nproc_per_node=2 {CODE_DIR}/train.py " \
          f"--data_dir {DATA_DIR} --out_dir {SM}"
    gpu0 = {"CUDA_VISIBLE_DEVICES": "0"}
    # A) full evaluation (val + 479 dev examples) + checkpoint upload to HF            [required]
    run(f"{py} --run_name smoke_llama --arch llama --micro_bs 8 --batch_tokens 8192 --max_steps 5 "
        f"--eval_every 5 --log_every 5 --hf_repo {CKPT_REPO}", gpu0)
    # B) resume FROM HUGGING FACE: local copy deleted, so last.pt must come from the hub  [required]
    shutil.rmtree(f"{SM}/smoke_llama")
    run(f"{py} --run_name smoke_llama --arch llama --micro_bs 8 --batch_tokens 8192 --max_steps 8 "
        f"--eval_every 100000 --log_every 1 --hf_repo {CKPT_REPO}", gpu0)
    # C) both GPUs, DistributedDataParallel, gradient accumulation like the main run    [required]
    run(f"{ddp} --run_name speed_ddp --arch llama --micro_bs 8 --batch_tokens 65536 --max_steps 6 "
        f"--eval_every 100000 --log_every 2")
    # D) torch.compile on one GPU                                                         [optional]
    run(f"{py} --run_name speed_compile --arch llama --micro_bs 8 --batch_tokens 8192 --max_steps 60 "
        f"--eval_every 100000 --log_every 10 --compile 1", gpu0, required=False)
    # E) torch.compile on both GPUs                                                       [optional]
    run(f"{ddp} --run_name speed_ddp_compile --arch llama --micro_bs 8 --batch_tokens 65536 --max_steps 8 "
        f"--eval_every 100000 --log_every 2 --compile 1", required=False)
    shutil.rmtree(SM, ignore_errors=True)
    print("\nSMOKE CHECKS DONE - paste this whole output to Claude.")

In [6]:
# ===================== NB2 CELL 6 - Architecture exploration (runs only when MODE == "explore") =====================
# 4 architectures, same tokenizer / data order / token budget / optimiser. Two runs at a time, one per T4.
# Each run checkpoints to HF; if the session dies, "Save & Run All" again resumes every unfinished run.
def tail_status(path):
    if not os.path.exists(path):
        return "(not started)"
    lines = [l for l in open(path).read().splitlines() if " step " in l or "EVAL" in l or "COMPLETE" in l
             or "Error" in l or "STOPPING" in l]
    return lines[-1][:200] if lines else "(starting)"

if MODE == "explore":
    RUNS = [("explore_llama", "llama"), ("explore_shared", "shared"), ("explore_gpt2", "gpt2"),
            ("explore_encdec", "encdec")]
    todo = queue.Queue()
    for r in RUNS:
        todo.put(r)
    def worker(gpu):
        while True:
            try:
                name, arch = todo.get_nowait()
            except queue.Empty:
                return
            cmd = [sys.executable, f"{CODE_DIR}/train.py", "--run_name", name, "--arch", arch,
                   "--data_dir", DATA_DIR, "--out_dir", OUT_DIR, "--hf_repo", CKPT_REPO,
                   "--micro_bs", "8", "--batch_tokens", "131072", "--max_tokens", str(EXPLORE_TOKENS),
                   "--mix", MIX, "--lr", "6e-4", "--eval_every", "100", "--log_every", "20",
                   "--ckpt_minutes", "60", "--time_limit_hours", "11", "--compile", "1"]
            with open(f"{OUT_DIR}/{name}.out", "a") as f:
                f.write(f"\n===== launching on GPU {gpu}: {' '.join(cmd)}\n"); f.flush()
                subprocess.run(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), stdout=f,
                               stderr=subprocess.STDOUT)
    threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
    for t in threads:
        t.start()
    t0 = time.time()
    while any(t.is_alive() for t in threads):
        for t in threads:
            t.join(timeout=600 / len(threads))
        print(f"--- {(time.time()-t0)/3600:.2f}h ---", flush=True)
        for name, _ in RUNS:
            print(f"  {name:16s} {tail_status(f'{OUT_DIR}/{name}.out')}", flush=True)

    print("\n================ EXPLORATION RESULTS ================")
    print(f"{'run':16s} {'params':>8s} {'dev_loss':>9s} {'dev_ppl':>8s} {'val_SC':>8s} {'val_avg':>8s} {'tok/s':>8s}")
    for name, arch in RUNS:
        d = f"{OUT_DIR}/{name}"
        if not os.path.exists(f"{d}/DONE"):
            print(f"{name:16s} NOT FINISHED - see {OUT_DIR}/{name}.out"); continue
        res = json.load(open(f"{d}/DONE"))
        cfgj = json.load(open(f"{d}/config.json")) if os.path.exists(f"{d}/config.json") else {}
        tps = [json.loads(l)["tok_s"] for l in open(f"{d}/log.jsonl") if '"tok_s"' in l]
        print(f"{name:16s} {cfgj.get('params', 0)/1e6:7.1f}M {res['dev_loss']:9.4f} {res['dev_ppl']:8.2f} "
              f"{res.get('val_supreme_court', float('nan')):8.4f} {res.get('val_avg', float('nan')):8.4f} "
              f"{(sum(tps[-10:])/max(1,len(tps[-10:]))):8,.0f}")

--- 0.17h ---
  explore_llama    [14:01:46][explore_llama] step 80/762 loss 5.2416 lr 5.98e-04 gnorm 0.70 20,899 tok/s mem 6.6GB elapsed 0.15h ETA 1.19h
  explore_shared   [14:02:00][explore_shared] step 80/762 loss 6.5329 lr 5.98e-04 gnorm 0.78 20,172 tok/s mem 5.8GB elapsed 0.16h ETA 1.23h
  explore_gpt2     (not started)
  explore_encdec   (not started)
--- 0.33h ---
  explore_llama    [14:12:27][explore_llama] step 180/762 loss 4.1354 lr 5.57e-04 gnorm 0.66 20,891 tok/s mem 6.6GB elapsed 0.33h ETA 1.01h
  explore_shared   [14:10:58][explore_shared] step 160/762 loss 5.5773 lr 5.69e-04 gnorm 0.93 19,850 tok/s mem 5.8GB elapsed 0.30h ETA 1.10h
  explore_gpt2     (not started)
  explore_encdec   (not started)
--- 0.50h ---
  explore_llama    [14:21:03][explore_llama] step 260/762 loss 3.7380 lr 4.93e-04 gnorm 0.54 20,899 tok/s mem 6.6GB elapsed 0.47h ETA 0.87h
  explore_shared   [14:22:12][explore_shared] step 260/762 loss 4.9475 lr 4.93e-04 gnorm 0.90 19,773 tok/s mem 5.8GB elapsed 0

In [7]:
# ===================== NB2 CELL 7 - MAIN training run on both GPUs (runs only when MODE == "main") =====================
# Stops itself after ~11h and saves; "Save & Run All" again continues from the exact same step.
def stream(cmd, env_extra=None):
    p = subprocess.Popen(cmd, shell=True, env=dict(os.environ, **(env_extra or {})), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()

MAIN_COMPILE = 1     # smoke checks D and E passed: +17-30% speed

if MODE == "main":
    ngpu = torch.cuda.device_count()
    launcher = (f"{sys.executable} -m torch.distributed.run --standalone --nproc_per_node={ngpu}" if ngpu > 1
                else sys.executable)
    code = stream(f"{launcher} {CODE_DIR}/train.py --run_name {MAIN_RUN} --arch {MAIN_ARCH} "
                  f"--data_dir {DATA_DIR} --out_dir {OUT_DIR} --hf_repo {CKPT_REPO} "
                  f"--micro_bs 8 --batch_tokens 262144 --max_tokens {MAIN_TOKENS} --mix {MIX} --lr 6e-4 "
                  f"--eval_every 250 --log_every 20 --ckpt_minutes 40 --time_limit_hours 11.0 --compile {MAIN_COMPILE} "
                  f"2>&1 | tee -a {OUT_DIR}/{MAIN_RUN}.out")
    print("exit code:", code)
    done = os.path.exists(f"{OUT_DIR}/{MAIN_RUN}/DONE")
    print("MAIN RUN FINISHED" if done else "Session limit reached - run 'Save & Run All' again to continue.")